## Setup

In [3]:
import pandas as pd
import numpy as np

## Code

In [4]:
ds = pd.read_csv('../data/raw/diabetes_risk.csv')
duplicateCount = ds.duplicated().sum()
emptyCount = ds.isnull().all(axis = 1).sum()

new_ds = ds.drop_duplicates().dropna(how = 'all')


print(f"Duplicate rows: {duplicateCount}")
print(f"Empty rows: {emptyCount}")
print(f"Original: {len(ds)}")
print(f"Cleaned: {len(new_ds)}")


Duplicate rows: 0
Empty rows: 0
Original: 15000
Cleaned: 15000



* Các column: patient_id,age,gender,city,bmi,family_history_diabetes,physical_activity_level,diet_type,smoking_status,alcohol_consumption,hours_sleep_per_night,stress_level,fasting_blood_sugar,hba1c_level,blood_pressure_systolic,blood_pressure_diastolic,waist_circumference_cm,income_bracket,diabetes_risk

* Các column nên xóa: city, physical_activity_level,diet_type,smoking_status,alcohol_consumption,hours_sleep_per_night,stress_level, income_bracket
=> Các column sẽ có: 
patient_id,age,gender,bmi,family_history_diabetes,fasting_blood_sugar,hba1c_level,blood_pressure_systolic,blood_pressure_diastolic,waist_circumference_cm,diabetes_risk

* Lý do: Đây là các chỉ số có tác động rất nhỏ cho việc dự đoán bệnh tiểu đường đặc biệt so với các chỉ số như bmi, huyết áp, etc. Đồng thời dataset đang có 19 biến khác nhau, giảm đi 8 biến sẽ có lợi rất lớn cho tốc độ huấn luyện và tiết kiệm tài nguyên tính toán.
* Các dòng rỗng, duplicate thì nên xóa để tránh các trường hợp đếm sai số lượng dòng trong data

In [ ]:
numerical_ds = ds.select_dtypes(include = [np.number])
correlationMatrix = numerical_ds.corr()
print('\n Tính tương quan giữa các biến số')

print('\n Tính tương quan giữa giới tính và khả năng tiểu đường')
crossTable1 = pd.crosstab(ds['gender'],ds['diabetes_risk'], margins=True, normalize='index') * 100
print(crossTable1)

print('\n Tính tương quan giữa lịch sử gia đình và khả năng tiểu đường')
crossTable2 = pd.crosstab(ds['family_history_diabetes'],ds['diabetes_risk'], margins=True, normalize='index') * 100
print(crossTable2)

print('\n Thống kê theo nhóm cho các tổ hợp quan trọng')
print('\n Giữa giới tính và khả năng tiểu đường')
groupData1 = ds.groupby(['diabetes_risk', 'gender']).agg(
    patientCount = ('patient_id', 'count'),
    avgBMI = ('bmi', 'mean'),
    avgHBA1C = ('hba1c_level', 'mean'), 
    avgSugar=('fasting_blood_sugar', 'mean',),  
    avgPressure=('blood_pressure_systolic', 'mean',),
)
groupData1 = groupData1.round(2)
print(groupData1)

print('\n Giữa lịch sử gia đình và khả năng tiểu đường')
groupData2 = ds.groupby(['diabetes_risk', 'family_history_diabetes']).agg(
    patientCount = ('patient_id', 'count'),
    avgBMI = ('bmi', 'mean'),
    avgHBA1C = ('hba1c_level', 'mean'), 
    avgSugar=('fasting_blood_sugar', 'mean',),  
    avgPressure=('blood_pressure_systolic', 'mean',),
)

groupData2 = groupData2.round(2)
print(groupData2)


Correlation between gender and diabetes risk
diabetes_risk       High        Low   Moderate
gender                                        
Female         15.216483  59.696506  25.087011
Male           14.731211  60.281837  24.986952
Other          18.300654  60.130719  21.568627
All            15.000000  60.000000  25.000000

Correlation between family history and diabetes risk
diabetes_risk                 High        Low   Moderate
family_history_diabetes                                 
No                       12.464125  65.856909  21.678967
Yes                      19.717773  49.103738  31.178490
All                      15.000000  60.000000  25.000000
